<a href="https://colab.research.google.com/github/pomellonn/strikt/blob/main/thesis_coverage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Расчёт смыслового покрытия статьи выбранными тезисами

In [ ]:
!pip install pymorphy3 natasha rouge-score razdel scikit-learn ruwordnet
!python3 -m ruwordnet download

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 57.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 100.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c3beb2fc5de53a759cd81c1ac592c476889c139763f511e9733770bdeab94199
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
  Created wheel for r

## Импорт и инициализация

In [ ]:
import re
import html
import ast

import numpy as np
import pandas as pd

from rouge_score import rouge_scorer


import pymorphy3
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, NewsNERTagger, Doc
from ruwordnet import RuWordNet
wn = RuWordNet()

morph = pymorphy3.MorphAnalyzer()

segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
ner_tagger = NewsNERTagger(emb)

## 1. Препроцессинг

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\u2029", " ").replace("\u2028", " ")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_article(text):
    text = clean_text(text)
    raw_parts = re.split(r"(?<=[.!?])\s+(?=[А-ЯЁA-Z«\"(])", text)
    merged = []
    for part in raw_parts:
        part = part.strip()
        if not part:
            continue
        if merged and merged[-1].count("«") > merged[-1].count("»"):
            merged[-1] = merged[-1] + " " + part
        else:
            merged.append(part)
    return merged


def tokenize(text):
    text = html.unescape(text).lower()
    return re.findall(r"[а-яёa-z0-9]+", text)


def lemmatize_text(text):
    return [morph.parse(t)[0].normal_form for t in tokenize(text)]


def lemmas_to_text(lemmas):
    return " ".join(lemmas)

## Модуль 1 — Lexical baseline (Jaccard + ROUGE + char n-grams)

In [ ]:
def jaccard_similarity(text1, text2):
    set1, set2 = set(lemmatize_text(text1)), set(lemmatize_text(text2))
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)


class WhitespaceTokenizer:
    def tokenize(self, text):
        return text.split()


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=WhitespaceTokenizer(),
)


def rouge_similarity(text1, text2):
    lemmas1 = lemmas_to_text(lemmatize_text(text1))
    lemmas2 = lemmas_to_text(lemmatize_text(text2))
    if not lemmas1 or not lemmas2:
        return 0.0
    scores = rouge.score(lemmas1, lemmas2)
    return np.mean([
        scores["rouge1"].fmeasure,
        scores["rouge2"].fmeasure,
        scores["rougeL"].fmeasure,
    ])


def char_ngrams(text, n=3):
    text = re.sub(r"[^а-яёa-z0-9 ]", "", html.unescape(text).lower())
    text = re.sub(r"\s+", " ", text)
    return {text[i:i + n] for i in range(len(text) - n + 1)}


def char_ngram_similarity(text1, text2, n=3):
    g1, g2 = char_ngrams(text1, n), char_ngrams(text2, n)
    if not g1 or not g2:
        return 0.0
    return len(g1 & g2) / len(g1 | g2)


def lexical_similarity(text1, text2, weights=(0.4, 0.3, 0.3)):
    w_rouge, w_jaccard, w_char = weights
    r = rouge_similarity(text1, text2)
    j = jaccard_similarity(text1, text2)
    c = char_ngram_similarity(text1, text2)
    return {
        "rouge": r, "jaccard": j, "char_ngram": c,
        "score": w_rouge * r + w_jaccard * j + w_char * c,
    }

In [ ]:
def calculate_part_coverage(part, selected_theses):
    best_score, best_claim, best_details = 0.0, None, None
    for claim in selected_theses:
        details = lexical_similarity(part, claim)
        if details["score"] > best_score:
            best_score, best_claim, best_details = details["score"], claim, details

    if best_details is None:
        best_details = {"rouge": 0.0, "jaccard": 0.0, "char_ngram": 0.0}

    return {
        "part": part, "best_claim": best_claim, "score": best_score,
        **{k: v for k, v in best_details.items() if k != "score"},
    }


def lexical_coverage(article_text, selected_theses):
    parts = split_article(article_text)
    if not parts or not selected_theses:
        return {"coverage": 0.0, "parts": []}

    results = [calculate_part_coverage(p, selected_theses) for p in parts]
    scores = np.array([r["score"] for r in results])
    coverage = float(scores.mean())
    return {"coverage": coverage, "parts": results}

## Модуль 2 — Entities + числа + даты + отрицания + RuWordNet

In [ ]:
NUMBER_PATTERN = re.compile(
    r"""
    (?<!\w)\d+(?:[.,]\d+)?\s*(?:%|процент(?:а|ов)?)
    |
    (?<!\w)\d+(?:[.,]\d+)?(?!\w)
    """,
    re.IGNORECASE | re.VERBOSE,
)

def extract_numbers(text):
    return [m.group(0).strip() for m in NUMBER_PATTERN.finditer(text)]

def normalize_number(value):
    return value.lower().replace(" ", "").replace(",", ".")

def number_set(text):
    return {normalize_number(x) for x in extract_numbers(text)}


MONTHS = "января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря"
DATE_PATTERN = re.compile(
    rf"""
    \b(?:
        \d{{1,2}}\s+(?:{MONTHS})(?:\s+\d{{4}})?
        |
        \d{{1,2}}[./-]\d{{1,2}}[./-]\d{{2,4}}
        |
        (?:19|20)\d{{2}}\s+г(?:ода)?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE,
)

def extract_dates(text):
    return [m.group(0).strip() for m in DATE_PATTERN.finditer(text)]

def date_set(text):
    return {d.lower().strip() for d in extract_dates(text)}


NEGATION_WORDS = {"не", "нет", "никогда", "никто", "ничего", "нигде", "нельзя", "без"}

def has_negation(text):
    return any(t in NEGATION_WORDS for t in tokenize(text))


def extract_concepts(text):
    concepts = []
    for token in tokenize(text):
        parsed = morph.parse(token)[0]
        if parsed.tag.POS in {"NOUN", "VERB", "ADJF", "ADJS"}:
            concepts.append(parsed.normal_form)
    return concepts


def extract_entities(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.tag_ner(ner_tagger)
    return {(span.text.lower().strip(), span.type) for span in doc.spans}

In [ ]:
def get_synonyms(word):
    synonyms = {word}
    if wn is None:
        return synonyms
    try:
        for sense in wn.get_senses(word):
            if sense.synset is not None and hasattr(sense.synset, 'lemmas'):
                for syn in sense.synset.lemmas:
                    synonyms.add(syn.name.lower().strip())
    except Exception:
        pass
    return synonyms


def expand_concepts(concepts):
    expanded = set()
    for c in concepts:
        expanded.add(c)
        expanded |= get_synonyms(c)
    return expanded

In [ ]:
def conceptual_coverage(article_text, selected_theses, penalty=0.1):
    article_text = clean_text(article_text)
    parts = split_article(article_text)
    if not parts or not selected_theses:
        return {"coverage": 0.0}

    theses_text = " ".join(selected_theses)

    article_numbers = number_set(article_text)
    theses_numbers = number_set(theses_text)
    num_matched = article_numbers & theses_numbers
    num_coverage = len(num_matched) / len(article_numbers) if article_numbers else 1.0

    article_dates = date_set(article_text)
    theses_dates = date_set(theses_text)
    date_matched = article_dates & theses_dates
    date_coverage = len(date_matched) / len(article_dates) if article_dates else 1.0

    article_entities = extract_entities(article_text)
    theses_entities = extract_entities(theses_text)
    ent_matched = article_entities & theses_entities
    ent_coverage = len(ent_matched) / len(article_entities) if article_entities else 1.0

    article_concepts = set(extract_concepts(article_text))
    theses_concepts_raw = set(extract_concepts(theses_text))
    theses_concepts_expanded = expand_concepts(theses_concepts_raw)
    con_matched = article_concepts & theses_concepts_expanded
    con_coverage = len(con_matched) / len(article_concepts) if article_concepts else 1.0

    factual_coverage = 0.45 * num_coverage + 0.2 * date_coverage + 0.35 * ent_coverage
    conceptual_coverage = con_coverage

    base_coverage = 0.5 * factual_coverage + 0.5 * conceptual_coverage
    number_conflict = bool(article_numbers and theses_numbers and not num_matched)

    entity_conflict = bool(article_entities and theses_entities and not ent_matched)

    article_negated_parts = [p for p in parts if has_negation(p)]
    negation_conflict = False
    if article_negated_parts:
        matching_theses_negation = [has_negation(t) for t in selected_theses]
        negation_conflict = not any(matching_theses_negation) and bool(con_matched)

    n_conflicts = int(number_conflict) + int(entity_conflict) + int(negation_conflict)
    final_coverage = float(np.clip(base_coverage - penalty * n_conflicts, 0, 1))

    return {
        "coverage": final_coverage,
        "factual_coverage": factual_coverage,
        "conceptual_coverage": conceptual_coverage,
    }

## Применение на датасете

In [ ]:
def parse_theses(theses_data):
    return theses_data if isinstance(theses_data, list) else ast.literal_eval(theses_data)

def get_selected_theses(all_theses, selected_indices):
    if isinstance(all_theses, str):
        all_theses = parse_theses(all_theses)
    if isinstance(selected_indices, str):
        selected_indices = ast.literal_eval(selected_indices)
    return [all_theses[i] for i in selected_indices]

In [ ]:
df = pd.read_csv('articles.csv')

In [ ]:
results = []

for i, row in df.iterrows():
    theses_all = parse_theses(row['theses'])
    selected_indices = ast.literal_eval(row['selected_indices'])
    selected = get_selected_theses(theses_all, selected_indices)

    lex = lexical_coverage(row['article'], selected)
    con = conceptual_coverage(row['article'], selected)

    results.append({
        'row': i,
        'method_1': lex['coverage'],
        'factual': con['factual_coverage'],
        'conceptual': con['conceptual_coverage'],
        'method_2': con['coverage'],
    })

results_df = pd.DataFrame(results)

In [ ]:
results

[{'row': 0,
  'method_1': 0.28468530485861904,
  'factual': 0.578125,
  'conceptual': 0.26666666666666666,
  'method_2': 0.4223958333333333},
 {'row': 1,
  'method_1': 0.2675587329728457,
  'factual': 0.0,
  'conceptual': 0.23655913978494625,
  'method_2': 0.11827956989247312},
 {'row': 2,
  'method_1': 0.0847804586351801,
  'factual': 0.11660691421254801,
  'conceptual': 0.06406685236768803,
  'method_2': 0.09033688329011802},
 {'row': 3,
  'method_1': 0.14060021890580235,
  'factual': 0.40208333333333335,
  'conceptual': 0.13043478260869565,
  'method_2': 0.1662590579710145},
 {'row': 4,
  'method_1': 0.20342274676184038,
  'factual': 0.10416666666666666,
  'conceptual': 0.21052631578947367,
  'method_2': 0.15734649122807015},
 {'row': 5,
  'method_1': 0.15453207521167756,
  'factual': 0.1279893924783028,
  'conceptual': 0.21171171171171171,
  'method_2': 0.16985055209500727},
 {'row': 6,
  'method_1': 0.21257649873029186,
  'factual': 0.23939393939393938,
  'conceptual': 0.218181818

In [ ]:
metrics_df = pd.DataFrame(results)

merged_df = df.join(metrics_df.set_index('row'))

merged_df.to_csv('articles_with_cov.csv', index=False, encoding='utf-8')